In [1]:
!pip install pyspark


# Инициализация сессии

In [2]:
from pyspark import SparkContext, SparkConf
import pyspark.sql as sql
from pyspark.sql import SparkSession
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf, col, max, sum, countDistinct

In [3]:
spark = SparkSession \
    .builder \
    .appName("L1_interactive_bike_analysis") \
    .getOrCreate()

In [4]:
spark.version

'4.0.2'

# Загрузка данных

In [8]:
import os
data_path = os.path.join(os.curdir, "data")
trips_path    = "/content/trip.csv"
stations_path = "/content/station.csv"

In [9]:
trip_data = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv(trips_path)

print("Trips")
trip_data.printSchema()

stations_data = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.option("timestampFormat", 'M/d/y H:m')\
.csv(stations_path)

print("Stations")
stations_data.printSchema()

Trips
root
 |-- id: integer (nullable = true)
 |-- duration: integer (nullable = true)
 |-- start_date: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: integer (nullable = true)
 |-- end_date: timestamp (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: integer (nullable = true)
 |-- bike_id: integer (nullable = true)
 |-- subscription_type: string (nullable = true)
 |-- zip_code: string (nullable = true)

Stations
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- dock_count: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- installation_date: string (nullable = true)



# Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):
1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

# 1 задание
Найти велосипед с максимальным временем пробега

In [15]:
# Группировка по id велосипеда и применение функции sum для подсчета времени пробега каждого велосипеда
max_trips_duration_per_bike = trip_data.groupBy("bike_id").agg(sum(col("duration")).alias("total_trips_duration"))

# Выбор велосипеда с максимальным пробегом
bike_with_max_trips_duration = max_trips_duration_per_bike.orderBy(col("total_trips_duration").desc()).first()

bike_id_with_max_duration = bike_with_max_trips_duration["bike_id"]
total_duration = bike_with_max_trips_duration["total_trips_duration"]

print(f"Велосипед {bike_id_with_max_duration} с суммарным временем пробега  = {total_duration}")

Велосипед 535 с суммарным временем пробега  = 18611693


# 2 задание
Найти наибольшее геодезическое расстояние между станциями

In [11]:
from math import sin, cos, sqrt, atan2, radians

def geodesic_distance(lat1, lon1, lat2, lon2):
    # Радиус Земли в километрах
    R = 6373.0
    lat1 = radians(lat1)
    lat2 = radians(lat2)
    lon1 = radians(lon1)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Вычисление геодезического расстояния по формуле Хаверсина
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# Конвертация функции в pyspark.sql.functions.udf (user-defined function)
geodesic_distance_udf = udf(geodesic_distance, DoubleType())

# Объединение датасета станций с самим собой для получения всех возможных пар
station_pairs = stations_data.alias("station1").crossJoin(stations_data.alias("station2"))

# Вычисление расстояния для каждой пары станций с помощью объявленной ранее функции
station_pairs_with_distance = station_pairs.withColumn(
    "geodesic_distance",
    geodesic_distance_udf(
        col("station1.lat"),
        col("station1.long"),
        col("station2.lat"),
        col("station2.long")
    )
)

# Поиск максимального геодезического расстояния среди всех расстояний для каждой пары станций
max_distance = station_pairs_with_distance.selectExpr("max(geodesic_distance) as max_distance").collect()[0]["max_distance"]

print(f"Максимальное геодезическое расстояние между станциями равно {max_distance} километрам")

Максимальное геодезическое расстояние между станциями равно 69.9428256877473 километрам


# 3 задание
Найти путь велосипеда с максимальным временем пробега через станции

In [18]:
# bike_id_with_max_duration = 535, найден в задании 1
bike_path = (
    trip_data
    .filter(col("bike_id") == bike_id_with_max_duration)
    .select("start_date", "start_station_name", "end_station_name", "duration")
    .orderBy("start_date")
)

print(f"Маршрут велосипеда #{bike_id_with_max_duration} через станции:")
bike_path.show(truncate=False)


Маршрут велосипеда #535 через станции:
+-------------------+---------------------------------------------+----------------------------------------+--------+
|start_date         |start_station_name                           |end_station_name                        |duration|
+-------------------+---------------------------------------------+----------------------------------------+--------+
|2013-08-29 19:32:00|Post at Kearney                              |San Francisco Caltrain (Townsend at 4th)|1245    |
|2013-08-29 21:38:00|San Francisco Caltrain (Townsend at 4th)     |San Francisco Caltrain 2 (330 Townsend) |423     |
|2013-08-30 08:40:00|San Francisco Caltrain 2 (330 Townsend)      |Market at Sansome                       |842     |
|2013-08-30 09:10:00|Market at Sansome                            |2nd at South Park                       |498     |
|2013-09-01 12:58:00|2nd at Townsend                              |Davis at Jackson                        |1671    |
|2013-09-05 11:59

# 4 задание
Найти количество велосипедов в системе

In [13]:
# Группировка по id велосипеда и подсчет уникальных значений id
unique_bikes_count = trip_data.agg(countDistinct("bike_id").alias("bike_count")).collect()[0]["bike_count"]

print(f"Суммарное количество велосипедов: {unique_bikes_count}")

Суммарное количество велосипедов: 700


# 5 задание
Найти пользователей потративших на поездки более 3 часов

In [17]:
users_with_total_trip_time = (
    trip_data
    .groupBy("zip_code")
    .sum("duration")
    .withColumnRenamed("sum(duration)", "total_time")
)

users_with_total_trip_time.filter("total_time > 10800").show()


+--------+----------+
|zip_code|total_time|
+--------+----------+
|   94102|  19128021|
|   95134|    728023|
|   84606|     95145|
|   80305|    180906|
|   60070|     28919|
|   95519|     30303|
|   43085|     11670|
|   91910|     50488|
|   77339|     13713|
|   48063|     13755|
|   85022|     12682|
|    1090|     20391|
|    2136|     16010|
|   11722|     24331|
|   95138|    155365|
|   94610|   3630628|
|   94404|   3589350|
|   80301|    152189|
|   91326|     65885|
|   90742|     10965|
+--------+----------+
only showing top 20 rows
